# Garmin data exploration
Run `scripts/backfill.py --days 90` first. Each chart reads the local SQLite database; nothing here calls Garmin.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": 0.3})
DB = Path("..") / "data" / "db" / "training.sqlite"
con = sqlite3.connect(DB)
m = pd.read_sql("SELECT * FROM daily_metrics ORDER BY date", con, parse_dates=["date"]).set_index("date")
a = pd.read_sql("SELECT * FROM activities ORDER BY start_time", con, parse_dates=["start_time"])
print(len(m), "days of metrics,", len(a), "activities")

## 1. Data coverage
How many days have each signal. Gaps here mean the planner must handle missing data.

In [ ]:
cov = m.drop(columns=["source"]).notna().mean().mul(100).round(0).sort_values()
cov.plot.barh(title="% of days with data", figsize=(8, 4)); plt.show()

## 2. Sleep

In [ ]:
fig, ax = plt.subplots()
ax.bar(m.index, m["sleep_hours"], color="#8da0cb", label="hours")
ax.set_ylabel("hours")
ax2 = ax.twinx(); ax2.plot(m.index, m["sleep_score"], color="#e78ac3", marker="o", ms=3, label="score")
ax2.set_ylabel("sleep score"); ax2.grid(False)
ax.set_title("Sleep duration and score"); plt.show()

## 3. HRV
Last night against the weekly average. Sustained drops below the weekly line signal fatigue.

In [ ]:
ax = m[["hrv_last_night_ms", "hrv_weekly_avg_ms"]].plot(marker="o", ms=3, title="HRV (ms)")
plt.show()

## 4. Resting heart rate

In [ ]:
m["resting_hr"].plot(marker="o", ms=3, title="Resting HR (bpm)", color="#fc8d62"); plt.show()

## 5. Body Battery
Morning value, with the day's min/max range.

In [ ]:
fig, ax = plt.subplots()
ax.fill_between(m.index, m["body_battery_min"], m["body_battery_max"], alpha=0.2, label="day range")
ax.plot(m.index, m["body_battery_morning"], marker="o", ms=3, color="#66c2a5", label="morning")
ax.legend(); ax.set_title("Body Battery"); plt.show()

## 6. Weekly training load by sport
Sum of Garmin's per-activity training load.

In [ ]:
if len(a):
    wk = a.assign(week=a["start_time"].dt.to_period("W").dt.start_time)
    wk.pivot_table(index="week", columns="sport", values="training_load", aggfunc="sum", fill_value=0) \
      .plot.bar(stacked=True, title="Weekly training load", figsize=(11, 3.5))
    plt.show()

## 7. Activities and heart-rate availability
Rides recorded without HR (for example the Sigma ROX) show up as missing.

In [ ]:
if len(a):
    a["has_hr"] = a["avg_hr"].notna()
    display(a.groupby(["sport", "has_hr"]).size().unstack(fill_value=0))
    display(a[["start_time", "sport", "source_sport", "duration_s", "avg_hr", "training_load"]].tail(15))